# Group 32 expanded multi-edit experiment
Run every subject in Arin's `groups_v2.json` in forward and reverse order. The catalog contains 141 subjects with two, three, or four edits each.

In Colab, choose **Runtime → Change runtime type** and select a GPU. The pilot was verified on an A100. Run the cells in order and authorize Google Drive when prompted. The setup downloads the project, dependencies, and experiment inputs as needed.

If Colab disconnects, reconnect with the same GPU type, rerun setup and input preparation, and run the full experiment cell again. The runner checks that the saved environment matches and reuses completed results.

In [ ]:
import json
from pathlib import Path
import shutil
import subprocess
import sys

import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before proceeding.'
print('GPU:', torch.cuda.get_device_name(0))
free_gib = shutil.disk_usage('/content').free / 2**30
print(f'Free disk: {free_gib:.1f} GiB')
assert free_gib > 20, 'Setup needs at least 20 GiB of free runtime disk.'

def run(command, cwd=None):
    with subprocess.Popen(
        [str(arg) for arg in command], cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            return_code = process.wait()
            if return_code:
                raise subprocess.CalledProcessError(return_code, process.args)
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise


## Install the experiment environment
The checkout is pinned to the merge on `main` that includes Arin's catalog and the verified runner. Model editing runs in the project's separate Python 3.11 environment.

In [ ]:
REPO_URL = 'https://github.com/Gogireddy007/Multifact-Rome.git'
SOURCE_COMMIT = 'bbd7320b2495b34e093eb7f1e71ce4a39b21ef8f'
ROOT = Path('/content/group32_expanded_multi')
if not ROOT.exists():
    run(['git', 'clone', '--branch', 'main', REPO_URL, ROOT])
changes = subprocess.check_output(
    ['git', 'status', '--porcelain', '--untracked-files=no'], cwd=ROOT, text=True,
).strip()
assert not changes, 'The experiment checkout has local edits. Use a fresh Colab runtime.'
run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=ROOT)
run([sys.executable, ROOT / 'scripts/setup.py'], cwd=ROOT)
PYTHON = ROOT / '.venv/bin/python'


## Prepare the expanded catalog and Drive output
The runner accepts a list of groups. This cell extracts that list from Arin's catalog and saves the original catalog alongside the results, where you can look up Arin's IDs and labels. Use the same Drive folder to resume a run.

In [ ]:
from collections import Counter
from google.colab import drive

catalog_path = ROOT / 'results/related_groups/groups_v2.json'
catalog = json.loads(catalog_path.read_text(encoding='utf-8'))
groups = catalog['groups']
assert groups, 'The expanded group catalog is empty.'
GROUP_COUNT = len(groups)
EXPECTED_RESULTS = 2 * GROUP_COUNT
print('Groups by edit count:', dict(sorted(Counter(len(g['case_ids']) for g in groups).items())))
print(f'Total: {GROUP_COUNT} groups, {EXPECTED_RESULTS} group/order results.')

drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/Group32/ROME_final/expanded_groups_v2')
OUT.mkdir(parents=True, exist_ok=True)

source_file = OUT / 'source_commit.txt'
if source_file.exists():
    assert source_file.read_text().strip() == SOURCE_COMMIT, 'Drive folder belongs to another source commit.'
else:
    source_file.write_text(SOURCE_COMMIT + '\n', encoding='utf-8')

GROUPS = Path('/content/group32_expanded_groups_input.json')
GROUPS.write_bytes(json.dumps(groups, sort_keys=True).encode('utf-8'))
shutil.copyfile(catalog_path, OUT / catalog_path.name)

def stage(limit):
    run([PYTHON, ROOT / 'scripts/run_multi_stage.py', '--groups', GROUPS,
         '--limit', limit, '--order', 'both', '--output', OUT], cwd=ROOT)


## Check the first one and five subjects
Each stage saves and verifies its results. The full run reuses these completed groups.

In [ ]:
stage(1)
stage(min(5, GROUP_COUNT))


## Run the full experiment
The runner prints an ETA as it completes each group and order. For groups with three or four edits, reverse order means reversing the entire sequence.

In [ ]:
stage(GROUP_COUNT)


## Verify and export
The ZIP is saved next to the results folder in Google Drive after the verifier checks every result. The summary JSON, comparison CSV, raw scores, logs, and run metadata are included.

In [ ]:
run([PYTHON, ROOT / 'scripts/verify_multi_results.py', '--output', OUT,
     '--limit', GROUP_COUNT], cwd=ROOT)
summary_path = OUT / f'summary_{GROUP_COUNT}.json'
comparison_path = OUT / f'comparison_{GROUP_COUNT}.csv'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
assert summary['verified'] and summary['complete_all_groups']
assert summary['order'] == 'both' and summary['num_groups'] == GROUP_COUNT
assert summary['num_group_orders'] == EXPECTED_RESULTS
assert len(list(OUT.glob('group_*.json'))) == EXPECTED_RESULTS
assert comparison_path.is_file(), f'Missing export: {comparison_path.name}'

archive = shutil.make_archive(str(OUT.parent / f'{OUT.name}_verified'), 'zip', root_dir=OUT)
print(f'COMPLETE: {GROUP_COUNT} groups, {EXPECTED_RESULTS} verified group/order results.')
print('Summary:', summary_path)
print('Comparison:', comparison_path)
print('ZIP in Google Drive:', archive)
